# 02 — API Keys & Data Lifecycle (SDK-first)

Keys CRUD + raw-key auth proof + send/get/statistics/delete with vanish proof.

In [ ]:
# Cell 1: install
%pip install -q "nostradamus-ioto-sdk @ git+https://github.com/robotics-4-all/nostradamus-ioto-sdk.git@4f8e7fc7c28c859eefed9f7382d1b148a5c6eeb8" python-dotenv

In [ ]:
import os, random, time
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv
load_dotenv()
BASE_URL = os.getenv("CENOTOO_BASE_URL", "http://localhost:8000")
USERNAME = os.getenv("CENOTOO_USERNAME", "")
PASSWORD = os.getenv("CENOTOO_PASSWORD", "")
assert BASE_URL and USERNAME and PASSWORD, "SKIP: set CENOTOO_BASE_URL/USERNAME/PASSWORD"

def unique_name(p="nb"):
    return f"{p}_{time.strftime('%Y%m%d%H%M%S')}_{random.randint(1000,9999)}"

def wait_for(fn, pred, timeout=60, interval=2, desc="condition"):
    import time as _t
    end = _t.monotonic() + timeout
    last = None
    while _t.monotonic() < end:
        last = fn()
        try:
            if pred(last):
                return last
        except Exception:
            pass
        _t.sleep(interval)
    raise TimeoutError(f"timed out: {desc}: {last!r}")

In [ ]:
from nostradamus_ioto_sdk import NostradamusClient
from nostradamus_ioto_sdk.models import KeyType
from nostradamus_ioto_sdk.exceptions import ResourceNotFoundError, NostradamusError, APIError
admin = NostradamusClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)
project = admin.projects.create(name=unique_name("nb2proj"), description="nb02", tags=["notebook", "02"])
collection = admin.collections.create(project_id=project.project_id, name=unique_name("nb2coll"),
    description="nb02", tags=["notebook", "02"],
    collection_schema={"key": "K", "timestamp": "2026-01-01T00:00:00Z", "temperature": 20.0, "humidity": 50.0})
print("setup:", project.project_id, collection.collection_id)

In [ ]:
pid = project.project_id
read_key = admin.project_keys.create(pid, KeyType.READ)
write_key = admin.project_keys.create(pid, KeyType.WRITE)
print("keys created; raw write key:", write_key.api_key[:4] + "...")
listed = admin.project_keys.list(pid)
assert len(listed) >= 2
# api_key auth proof with the RAW secret
key_client = NostradamusClient(base_url=BASE_URL, api_key=write_key.api_key)
key_client.data.send(project_id=pid, collection_id=collection.collection_id,
    data={"key": unique_name("keyauth"), "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
           "temperature": 21.5, "humidity": 55.0})
print("api_key auth proof: send OK")
key_client.close()
regen = admin.project_keys.regenerate(pid, write_key.api_key)
assert regen.api_key != write_key.api_key
new_key = regen.api_key
print("regenerated OK")

In [ ]:
prefix = unique_name("data")
base = datetime.now(timezone.utc).replace(microsecond=0)
records = [{"key": f"{prefix}_{i:03d}", "timestamp": (base - timedelta(minutes=i)).isoformat().replace("+00:00", "Z"),
             "temperature": 20.0 + i, "humidity": 50.0 + i} for i in range(3)]
admin.data.send(project_id=pid, collection_id=collection.collection_id, data=records)
want = {r["key"] for r in records}
def _q():
    seen = set()
    for k in want:
        r = admin.data.get(project_id=pid, collection_id=collection.collection_id,
            filters=[{"property_name": "key", "operator": "eq", "property_value": k}], limit=10)
        if len(r.data) >= 1:
            seen.add(k)
    return seen
found = wait_for(_q, lambda s: want <= s, desc="3 records readable")
print("read-back OK:", sorted(found))
stats = admin.data.statistics(project_id=pid, collection_id=collection.collection_id, stat="count", attribute="temperature")
assert stats is not None
print("statistics OK")
try:
    ks = admin.collection_keys.get_statistics(project_id=pid, collection_id=collection.collection_id, key=records[0]["key"])
    assert ks.total_records >= 1
    print("per-key stats OK")
except APIError as e:
    if getattr(e, "status_code", None) == 500:
        print(f"SKIP per-key stats (server 500): {e}")
    else:
        raise

In [ ]:
admin.data.delete(project_id=pid, collection_id=collection.collection_id, key=records[0]["key"])
gone = wait_for(lambda: admin.data.get(project_id=pid, collection_id=collection.collection_id,
    filters=[{"property_name": "key", "operator": "eq", "property_value": records[0]["key"]}], limit=10),
    lambda r: len(r.data) == 0, desc="deleted record vanishes")
assert len(gone.data) == 0
print("delete/vanish proof OK")

In [ ]:
import contextlib
with contextlib.suppress(Exception):
    admin.project_keys.delete(pid, new_key)
with contextlib.suppress(Exception):
    admin.collections.delete(pid, collection.collection_id)
with contextlib.suppress(Exception):
    admin.projects.delete(pid)
admin.close()
print("cleanup done")